# FPP14 — Timing of the Deployed Pipeline: Accelerator (T4, proxy for a terminal NPU) and CPU vs FingerJetFX + BOZORTH3

## Target
1:1 verification on **recent attendance terminals with an NPU**. On such a terminal the neural models run on the NPU, while FingerJetFX + BOZORTH3 (classic C code) run on the CPU. Colab has no NPU; the closest available hardware for **one fingerprint at a time** is the **T4 GPU** (low-power inference GPU, fp16/int8 units). It is reported as an *accelerator proxy*, not as the terminal; the **CPU** numbers (1 thread = worst case, and all cores) are reported too.

## What is timed (one item per call = batch 1, as on the terminal)
| Path | Our system | Old system |
|---|---|---|
| **Daily login after migration: image vs image** (after re-enrolment) | letterbox to 224 (CPU) → image model (FPP05 winner = FPP04L EfficientNetV2-S) → cosine 1:1 | FingerJetFX extraction → ISO → xyt → BOZORTH3 1:1 |
| **First login** after migration (once per user) | letterbox → image model → Siamese image tower (FPP13 winner) → cosine with the migration template | — |
| **Migration** (once per stored template, offline batch) | minutiae set → minutiae encoder + Siamese tower (FPP13 winner) | — |

## Rules
- **Excluded from every time:** model building and weight loading, graph tracing / compilation, image decoding, and the first **warm-up** calls (30 per model; 10 for FingerJetFX; BOZORTH3 process start-up is excluded by timing many pairs in one call after a warm-up call).
- Each measurement runs in a **fresh process** (GPU hidden for the CPU runs, thread limits applied); every model is checked against its saved embeddings before timing (cosine ≥ 0.999 in fp32).
- Per call: mean, median, std, p95 over 300 real items (MOLF DB1, DB2, FingerDOS); each configuration measured **3 times** → mean ± std of the per-run means.
- Batch-64 throughput is reported separately (migration is an offline batch job).
- **Complexity:** parameters, FLOPs per item, model size (fp32, fp16).

**Runtime:** Colab **T4 GPU** (the CPU runs use the same machine with the GPU hidden). **Needs:** FPP05, FPP09 and FPP13 manifests, FPP02 templates/originals, FPP03 scores, fpp_lib ≥ 1.13.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 13): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.13')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration and the models to time (repetition 1; timing does not depend on the weights)

In [ ]:
import subprocess, shutil, matplotlib.pyplot as plt
KERNEL = 'FPP14'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'; FIG.mkdir(parents=True, exist_ok=True); WORK = Path('/content/fpp14'); WORK.mkdir(exist_ok=True)
REP = L.repetitions(ws)[0]
N_ITEMS = 300; WARMUP = 30; MEASURE_REPEATS = 3; BATCH = 64
def winner(kernel):
    mf = json.loads((P.kernel_dir(ws, kernel) / 'winner_manifest.json').read_text()); return {r: {k: str(Path(ws['output']) / v) for k, v in f.items()} for r, f in mf['files'].items()}, mf
img_files, mf5 = winner('FPP05'); mig_files, mf13 = winner('FPP13'); _, mf9 = winner('FPP09')
IMG_DIR = Path(img_files[REP]['pair_scores.npz']).parent; MIG_DIR = Path(mig_files[REP]['pair_scores.npz']).parent
need = [IMG_DIR / 'encoder.weights.h5', IMG_DIR / 'embeddings_float32.npz', MIG_DIR / 'encoder.weights.h5', MIG_DIR / 'minutiae_tower.weights.h5', MIG_DIR / 'image_tower.weights.h5',
        MIG_DIR / 'embeddings_float32.npz', MIG_DIR / 'image_side_embeddings_float32.npz', P.kernel_dir(ws, 'FPP13') / 'fpp13_minutiae.py']
miss = [str(p_) for p_ in need if not p_.is_file()]
if miss: raise FileNotFoundError(f'missing: {miss}')
GPU = bool(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).returncode == 0)
GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if GPU else 'none'
SPEC = dict(rep=REP, img_dir=str(IMG_DIR), image_model=json.loads((IMG_DIR / 'encoder_config.json').read_text()).get('backbone', 'EfficientNetV2S') if (IMG_DIR / 'encoder_config.json').is_file() else 'EfficientNetV2S',
            mig_dir=str(MIG_DIR), module=str(P.kernel_dir(ws, 'FPP13') / 'fpp13_minutiae.py'), minutiae_config=mf13.get('minutiae_config', mf9['config']), n_items=N_ITEMS, warmup=WARMUP, batch=BATCH)
(WORK / 'spec.json').write_text(json.dumps(SPEC, indent=1))
CONFIGS = ([dict(device='gpu', precision='fp32', threads=0), dict(device='gpu', precision='fp16', threads=0)] if GPU else []) + \
          [dict(device='cpu', precision='fp32', threads=1), dict(device='cpu', precision='fp32', threads=os.cpu_count())]
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], repetition=REP, image_model=f"{SPEC['image_model']} ({L.rel(ws, IMG_DIR)})", migration_matcher=L.rel(ws, MIG_DIR),
              gpu=GPU_NAME, cpu_count=os.cpu_count(), configs=CONFIGS, items_per_measure=N_ITEMS, warmup_calls_excluded=WARMUP, measure_repeats=MEASURE_REPEATS, batch_throughput=BATCH,
              excluded='model build + weight loading, graph tracing / compilation, image decoding, warm-up calls; BOZORTH3 start-up amortised', fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))
if not GPU: print('\nNo GPU in this runtime: only the CPU rows will be measured. Switch to a T4 GPU runtime for the accelerator rows.')

## 2. The timing worker (fresh process per configuration)
The worker rebuilds each deployed model exactly as trained, loads its weights, **checks it against the saved embeddings**, warms it up (excluded), then times one item per call.
The fp16 configuration applies to the image model only (mixed float16 on the T4); the migration encoder and the towers always run in fp32.

In [ ]:
WORKER = r"""
import os, sys, json, time, numpy as np
device, precision, threads, out = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
if device == 'cpu':
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = str(threads)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
if device == 'cpu': tf.config.threading.set_intra_op_parallelism_threads(threads); tf.config.threading.set_inter_op_parallelism_threads(threads)
else:
    for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
sys.path.insert(0, '/content'); import matplotlib; matplotlib.use('Agg'); import fpf_lib as L, h5py, importlib.util
S = json.load(open('/content/fpp14/spec.json')); ws = L.workspace(); IMAGE_SIZE = 224; EMBEDDING_DIM = 1024; DROPOUT = 0.30
spec = importlib.util.spec_from_file_location('fpp13_minutiae', S['module']); M = importlib.util.module_from_spec(spec); spec.loader.exec_module(M)
M_CFG = dict(M.SETUP, **S['minutiae_config']); Lyr = tf.keras.layers
# ---- migration matcher (always fp32): FPP13 minutiae encoder + Siamese tower
class ModalityOffset(Lyr.Layer):
    def build(self, shape): self.b = self.add_weight(name='offset', shape=(int(shape[-1]),), initializer='zeros')
    def call(self, x): return x + self.b
def build_siamese():
    xm = tf.keras.Input((M.EMBEDDING_DIM,), name='minutiae_embedding'); xi = tf.keras.Input((M.EMBEDDING_DIM,), name='image_embedding')
    l2 = lambda n: Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name=n)
    om = ModalityOffset(name='minutiae_offset'); oi = ModalityOffset(name='image_offset')
    hd = Lyr.Dense(2048, activation='gelu', name='shared_hidden'); dr = Lyr.Dropout(0.1, name='shared_dropout'); out_ = Lyr.Dense(M.EMBEDDING_DIM, name='shared_out')
    f = tf.keras.Model(xm, l2('minutiae_shared_embedding')(out_(dr(hd(om(xm))))), name='siamese_minutiae'); g = tf.keras.Model(xi, l2('image_shared_embedding')(out_(dr(hd(oi(xi))))), name='siamese_image')
    return f, g
md_ = S['mig_dir']; enc = M.build_ls_encoder(M_CFG); enc.load_weights(md_ + '/encoder.weights.h5')
ft, gt = build_siamese(); ft.load_weights(md_ + '/minutiae_tower.weights.h5'); gt.load_weights(md_ + '/image_tower.weights.h5')
MINU = M.load_minutiae(L.local_copy(ws['templates'] / 'MOLF_DB1_fingerjetfx_templates.h5'))
SETS = np.stack([M.to_set(m, quality=M_CFG['quality']) for m in MINU[:S['n_items']]])
# ---- image model (FPP04C / FPP04L architecture), fp32 or mixed fp16
if precision == 'fp16': tf.keras.mixed_precision.set_global_policy('mixed_float16')
A = tf.keras.applications
def build_image_model(name):
    bb = getattr(A, name)(include_top=False, weights=None, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), pooling='avg')
    image = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 1), dtype=tf.uint8, name='fingerprint_uint8')
    x = Lyr.Lambda(lambda z: tf.cast(z, tf.float32), name='cast_float')(image); x = Lyr.Concatenate(name='gray_to_rgb')([x, x, x])
    f = bb(x, training=False); f = Lyr.Dropout(DROPOUT, name='head_dropout')(f)
    e = Lyr.Dense(EMBEDDING_DIM, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(tf.cast(z, tf.float32), axis=1), name='embedding')(e)
    return tf.keras.Model(image, e, name=f'{name}_full_encoder')
img = build_image_model(S['image_model']); img.load_weights(S['img_dir'] + '/encoder.weights.h5')
X = []
for d in ('MOLF_DB1', 'MOLF_DB2', 'FingerDOS'):
    with h5py.File(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5'), 'r') as h: X.append(np.asarray(h['images'][:S['n_items'] // 3]))
X = np.concatenate(X); X = X[..., None] if X.ndim == 3 else X
f_img = tf.function(lambda z: img(z, training=False)); f_mig = tf.function(lambda z: ft(enc(z, training=False), training=False))
f_tow = tf.function(lambda z: gt(z, training=False)); f_first = tf.function(lambda z: gt(img(z, training=False), training=False))
# ---- correctness against the saved embeddings
l2n = lambda a: a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
Ei = L.load_npz(S['img_dir'] + '/embeddings_float32.npz')['MOLF_DB1'][:16]; Ws = L.load_npz(S['img_dir'] + '/embeddings_float32.npz')['MOLF_DB1'][:16]
c_img = float(np.min(np.sum(l2n(Ei) * f_img(tf.constant(X[:16])).numpy(), 1)))
Em = L.load_npz(md_ + '/embeddings_float32.npz')['MOLF_DB1'][:16]; c_mig = float(np.min(np.sum(l2n(Em) * f_mig(tf.constant(SETS[:16])).numpy(), 1)))
Eg = L.load_npz(md_ + '/image_side_embeddings_float32.npz')['MOLF_DB1'][:16]; c_tow = float(np.min(np.sum(l2n(Eg) * f_tow(tf.constant(l2n(Ei).astype(np.float32))).numpy(), 1)))
def timed(fn, data, name):
    for i in range(S['warmup']): fn(tf.constant(data[i % len(data)][None])).numpy()          # warm-up: excluded
    lat = []
    for i in range(len(data)):
        x = tf.constant(data[i][None]); t = time.perf_counter(); fn(x).numpy(); lat.append(1000 * (time.perf_counter() - t))
    lat = np.asarray(lat); xb = tf.constant(data[:S['batch']]); fn(xb).numpy(); fn(xb).numpy(); tb = []                  # batch warm-up: excluded
    for _ in range(10): t = time.perf_counter(); fn(xb).numpy(); tb.append(1000 * (time.perf_counter() - t) / len(xb))
    return dict(component=name, items=int(len(lat)), ms_mean=float(lat.mean()), ms_median=float(np.median(lat)), ms_std=float(lat.std()), ms_p95=float(np.percentile(lat, 95)),
                ms_min=float(lat.min()), batch_ms_per_item=float(np.median(tb)))
E_img = f_img(tf.constant(X[:64])).numpy().astype(np.float32)
rows = [timed(f_img, X, 'image model (probe image -> embedding)'), timed(f_first, X, 'image model + Siamese image tower (first login)'),
        timed(f_tow, E_img, 'Siamese image tower only'), timed(f_mig, SETS, 'migration: minutiae encoder + Siamese tower (per template)')]
g_ = l2n(np.random.default_rng(0).normal(size=(10000, EMBEDDING_DIM)).astype(np.float32)); q = g_[0].copy()
for i in range(1000): float(q @ g_[i])
t = time.perf_counter()
for i in range(20000): float(q @ g_[i % 10000])
rows.append(dict(component='cosine 1:1 (numpy, CPU)', items=20000, ms_mean=1000 * (time.perf_counter() - t) / 20000))
# ---- complexity (FLOPs per item, batch 1, fp32 graph)
def flops(fn, spec_):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        cf = convert_variables_to_constants_v2(fn.get_concrete_function(spec_)); o = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        return int(tf.compat.v1.profiler.profile(graph=cf.graph, run_meta=tf.compat.v1.RunMetadata(), cmd='op', options=o).total_float_ops)
    except Exception as exc: return None
cx = None
if precision == 'fp32' and device == 'cpu' and threads == 1:
    cx = [dict(model='image model', parameters=int(img.count_params()), flops_per_item=flops(tf.function(lambda z: img(z, training=False)), tf.TensorSpec((1, IMAGE_SIZE, IMAGE_SIZE, 1), tf.uint8))),
          dict(model='Siamese tower (shared by both sides)', parameters=int(gt.count_params()), flops_per_item=flops(tf.function(lambda z: gt(z, training=False)), tf.TensorSpec((1, M.EMBEDDING_DIM), tf.float32))),
          dict(model='minutiae encoder', parameters=int(enc.count_params()), flops_per_item=flops(tf.function(lambda z: enc(z, training=False)), tf.TensorSpec((1,) + SETS.shape[1:], tf.float32)))]
    for r in cx: r.update(size_mb_fp32=4 * r['parameters'] / 2 ** 20, size_mb_fp16=2 * r['parameters'] / 2 ** 20)
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')
gpu = ', '.join(d.name for d in tf.config.list_logical_devices('GPU'))
meta = dict(device=device, precision=precision, threads=threads, cpu_model=cpu, gpu_devices=gpu, tensorflow=tf.__version__,
            check_image_model=round(c_img, 5), check_migration=round(c_mig, 5), check_tower=round(c_tow, 5))
json.dump(dict(meta=meta, rows=rows, complexity=cx), open(out, 'w')); print(json.dumps(meta))
"""
(WORK / 'timing_worker.py').write_text(WORKER)
BZ_WORKER = r"""
import os, sys
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import subprocess, shutil, time, ctypes, tempfile, json
from pathlib import Path
sys.path.insert(0, '/content')
""" + 'import numpy as np, pandas as pd, h5py, cv2, struct\nimport matplotlib; matplotlib.use(\'Agg\'); import fpf_lib as L, fpp_lib as P\ncv2.setNumThreads(1)\nws = L.workspace()\nN_PER_DATASET = 100; WARMUP = 10; PAIRS_PER_SET = 700; SINGLE_CALLS = 100; MAX_MINUTIAE = 150; IMAGE_SIZE = 224\nNBIS_REPO = \'https://github.com/mike10004/nbis.git\'; NBIS_COMMIT = \'e2122d4508ab5cc4761fe29caa8a161539c177d8\'      # same pinned NBIS as FPP03\ndef run(cmd, cwd=None):\n    p = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=True)\n    if p.returncode != 0: raise RuntimeError(f\'{cmd}\\n{p.stdout[-2000:]}\\n{p.stderr[-2000:]}\')\n    return p.stdout\n\n# ---------------------------------------------------------------- FingerJetFX (library built by FPP02, same pinned commit)\nLIB = Path(\'/content/libFJFX.so\'); shutil.copy2(ws[\'templates\'] / \'libFJFX.so\', LIB)\nFMD_ISO = 0x01010001; FMD_BUF = 34 + 256 * 6\n_fjfx = ctypes.CDLL(str(LIB)).fjfx_create_fmd_from_raw\n_fjfx.argtypes = [ctypes.c_void_p, ctypes.c_ushort, ctypes.c_uint, ctypes.c_uint, ctypes.c_uint, ctypes.c_void_p, ctypes.POINTER(ctypes.c_uint)]; _fjfx.restype = ctypes.c_int\ndef extract_iso(gray, ppi):\n    img = np.ascontiguousarray(gray, np.uint8); h, w = img.shape; buf = (ctypes.c_ubyte * FMD_BUF)(); size = ctypes.c_uint(FMD_BUF)\n    err = _fjfx(img.ctypes.data_as(ctypes.c_void_p), int(ppi), h, w, FMD_ISO, ctypes.cast(buf, ctypes.c_void_p), ctypes.byref(size))\n    return int(err), (bytes(buf[:size.value]) if err == 0 else b\'\')\ndef parse_iso(data):\n    pos = 28; n = data[27]; x, y, t, q = [], [], [], []\n    for i in range(n):\n        a, b, ang, qq = struct.unpack(\'>HHBB\', data[pos:pos + 6]); pos += 6; x.append(a & 0x3FFF); y.append(b & 0x3FFF); t.append((360.0 - ang * 360.0 / 256.0) % 360.0); q.append(qq / 100.0)\n    return np.array(x, np.float32), np.array(y, np.float32), np.array(t, np.float32), np.array(q, np.float32)\ndef xyt_text(x, y, theta_img_deg, quality):     # identical to FPP03 write_xyt\n    iso = np.rint((360.0 - np.asarray(theta_img_deg, np.float64)) % 360.0).astype(int) % 360; q = np.rint(100 * np.asarray(quality)).astype(int)\n    return \'\'.join(f\'{int(round(float(x[i])))} {int(round(float(y[i])))} {iso[i]} {q[i]}\\n\' for i in np.argsort(-q, kind=\'stable\')[:MAX_MINUTIAE])\ndef letterbox(image):                            # identical to FPP02\n    h, w = image.shape; s = min(IMAGE_SIZE / w, IMAGE_SIZE / h); nw, nh = max(1, round(w * s)), max(1, round(h * s))\n    r = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)\n    c = np.full((IMAGE_SIZE, IMAGE_SIZE), 255, np.uint8); x0 = (IMAGE_SIZE - nw) // 2; y0 = (IMAGE_SIZE - nh) // 2; c[y0:y0 + nh, x0:x0 + nw] = r; return c\n\n# ---------------------------------------------------------------- BOZORTH3 (same pinned NBIS source and build line as FPP03)\nSRC = Path(\'/content/nbis_src\'); BZ = Path(\'/content/bozorth3_build\'); BZ.mkdir(exist_ok=True); BOZORTH3 = BZ / \'bozorth3\'\nif not BOZORTH3.is_file():\n    if not (SRC / \'.git\').is_dir(): run(f\'git clone -q {NBIS_REPO} {SRC}\')\n    run(f\'git checkout -q {NBIS_COMMIT}\', cwd=SRC)\n    (BZ / \'version.h\').write_text(\'#define VERSION "NBIS bozorth3 (FPP timing build)"\\n\'); (BZ / \'version_stub.c\').write_text(\'#include <stdio.h>\\nvoid getVersion(void){printf("bozorth3\\\\n");}\\n\')\n    run(f\'gcc -O2 -w -I{BZ} -Ibozorth3/include -Icommonnbis/include bozorth3/src/bin/bozorth3/bozorth3.c bozorth3/src/bin/bozorth3/usage.c bozorth3/src/lib/bozorth3/*.c {BZ}/version_stub.c -o {BOZORTH3} -lm\', cwd=SRC)\ndef bozorth3(pairs):\n    with tempfile.NamedTemporaryFile(\'w\', suffix=\'.lis\', delete=False) as f:\n        for a, b in pairs: f.write(f\'{a}\\n{b}\\n\')\n        lis = f.name\n    p = subprocess.run([str(BOZORTH3), \'-m1\', \'-M\', lis], text=True, capture_output=True); os.unlink(lis)\n    if p.returncode != 0: raise RuntimeError(p.stderr[-500:])\n    v = [int(t) for t in p.stdout.split()]; assert len(v) == len(pairs); return np.asarray(v, np.float64)\n\n# ---------------------------------------------------------------- 1) FingerJetFX extraction, xyt, letterbox: per image, on the same originals\nrows_img, check_bytes = [], []\nfor d in L.DATASETS:\n    with h5py.File(L.local_copy(ws[\'originals\'] / f\'{d}_originals_png.h5\'), \'r\') as h:\n        n = len(h[\'png\']); idx = np.linspace(0, n - 1, N_PER_DATASET).astype(int)\n        imgs = [cv2.imdecode(np.asarray(h[\'png\'][i]), cv2.IMREAD_GRAYSCALE) for i in idx]; ppi = h[\'ppi\'][:][idx]\n    with h5py.File(L.local_copy(ws[\'templates\'] / f\'{d}_fingerjetfx_templates.h5\'), \'r\') as h: stored = [bytes(np.asarray(h[\'minutiae/iso_template\'][i])) for i in idx]\n    for i in range(WARMUP): extract_iso(imgs[i % len(imgs)], ppi[i % len(imgs)])\n    for k, (img, pp) in enumerate(zip(imgs, ppi)):\n        t = time.perf_counter(); err, rec = extract_iso(img, pp); fx = 1000 * (time.perf_counter() - t)\n        t = time.perf_counter(); x, y, th, q = parse_iso(rec) if err == 0 else (np.zeros(0),) * 4; txt = xyt_text(x, y, th, q); p_ = Path(f\'/content/_t.xyt\'); p_.write_text(txt); cv = 1000 * (time.perf_counter() - t)\n        t = time.perf_counter(); letterbox(img); lb = 1000 * (time.perf_counter() - t)\n        check_bytes.append(rec == stored[k])\n        rows_img.append({\'dataset\': d, \'image\': int(idx[k]), \'width\': img.shape[1], \'height\': img.shape[0], \'ppi\': int(pp), \'minutiae\': int(len(x)),\n                         \'fingerjetfx_ms\': fx, \'iso_to_xyt_ms\': cv, \'letterbox_224_ms\': lb})\nI = pd.DataFrame(rows_img); same = float(np.mean(check_bytes))\nprint(f\'FingerJetFX output identical to the stored FPP02 templates: {100 * same:.1f} % of {len(check_bytes)} images\')\n\n# ---------------------------------------------------------------- 2) BOZORTH3 on real test pairs (repetition 1), scores checked against FPP03\nrep = L.repetitions(ws)[0]; proto, Pp = P.load_pairs(ws, rep); S03 = P.load_scores(P.kernel_dir(ws, \'FPP03\') / \'runs\' / rep / \'BOZORTH3\')\nXYT = {}\ndef xyt_file(d, i):\n    key = (d, int(i))\n    if key not in XYT:\n        g = H[d][\'minutiae\']; p_ = Path(f\'/content/xyt_timing/{d}/{int(i):06d}.xyt\'); p_.parent.mkdir(parents=True, exist_ok=True)\n        p_.write_text(xyt_text(g[\'x\'][i], g[\'y\'][i], g[\'theta\'][i], g[\'quality\'][i])); XYT[key] = str(p_)\n    return XYT[key]\nH = {d: h5py.File(L.local_copy(ws[\'templates\'] / f\'{d}_fingerjetfx_templates.h5\'), \'r\') for d in L.DATASETS}\nrows_bz, match = [], []\nfor name in (\'DB1_test\', \'DB2_test\', \'FingerDOS_test\'):\n    da, db, _ = P.PAIR_SETS[name]; p = Pp[name]; k = np.linspace(0, len(p[\'label\']) - 1, PAIRS_PER_SET).astype(int)\n    pairs = [(xyt_file(da, p[\'index_a\'][j]), xyt_file(db, p[\'index_b\'][j])) for j in k]\n    bozorth3(pairs[:50])                                                                   # warm-up (file cache)\n    t = time.perf_counter(); sc = bozorth3(pairs); per_pair = 1000 * (time.perf_counter() - t) / len(pairs)\n    single = []\n    for a, b in pairs[:SINGLE_CALLS]: t = time.perf_counter(); bozorth3([(a, b)]); single.append(1000 * (time.perf_counter() - t))\n    ref = S03[name][1][k]; match.append(float(np.mean(sc == ref)))\n    rows_bz.append({\'pair_set\': name, \'pairs\': len(pairs), \'bozorth3_ms_per_pair_amortised\': per_pair, \'bozorth3_ms_one_pair_per_call_mean\': float(np.mean(single)),\n                    \'bozorth3_ms_one_pair_per_call_median\': float(np.median(single)), \'scores_equal_to_FPP03\': float(np.mean(sc == ref))})\nfor h in H.values(): h.close()\nB = pd.DataFrame(rows_bz); print(f\'BOZORTH3 scores identical to the saved FPP03 scores: {100 * np.mean(match):.1f} %\')\n\n' + r"""
I.to_csv(sys.argv[1], index=False); B.to_csv(sys.argv[2], index=False)
json.dump({'fingerjetfx_identical_to_FPP02': same, 'bozorth3_scores_identical_to_FPP03': float(np.mean(match)),
           'cpu_model': next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')}, open(sys.argv[3], 'w'))
"""
(WORK / 'bozorth3_worker.py').write_text(BZ_WORKER); print('workers written to', WORK)

## 3. Run the measurements (each configuration 3 times, fresh process each time)

In [ ]:
RAW = []; META = []; CX = None
for cfg in CONFIGS:
    for k in range(MEASURE_REPEATS):
        out = WORK / f"t_{cfg['device']}_{cfg['precision']}_{cfg['threads']}_{k}.json"; t0 = time.time()
        r = subprocess.run([sys.executable, str(WORK / 'timing_worker.py'), cfg['device'], cfg['precision'], str(cfg['threads']), str(out)], capture_output=True, text=True)
        if r.returncode != 0: print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError(f'timing failed: {cfg}')
        d = json.load(open(out)); m = d['meta']; META.append({**m, 'measure': k})
        lim = 0.99 if cfg['precision'] == 'fp16' else 0.999
        if min(m['check_migration'], m['check_tower']) < 0.999 or m['check_image_model'] < lim: raise RuntimeError(f'model check failed (wrong weights or build): {m}')
        for row in d['rows']: RAW.append({**cfg, 'measure': k, **row})
        if d['complexity']: CX = d['complexity']
        print(f"{cfg} measure {k + 1}: ok ({time.time() - t0:.0f} s) | checks image {m['check_image_model']}, migration {m['check_migration']}, tower {m['check_tower']}")
RAW = pd.DataFrame(RAW); META = pd.DataFrame(META)
print('\nFingerJetFX + BOZORTH3 (single thread, CPU; 3 measures)')
BZI, BZB, BZC = [], [], []
for k in range(MEASURE_REPEATS):
    fi, fb, fc = WORK / f'fjfx_{k}.csv', WORK / f'bz_{k}.csv', WORK / f'chk_{k}.json'
    r = subprocess.run([sys.executable, str(WORK / 'bozorth3_worker.py'), str(fi), str(fb), str(fc)], capture_output=True, text=True)
    if r.returncode != 0: print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('FingerJetFX / BOZORTH3 timing failed')
    BZI.append(pd.read_csv(fi).assign(measure=k)); BZB.append(pd.read_csv(fb).assign(measure=k)); BZC.append(json.load(open(fc)))
    print(f'  measure {k + 1}: ok | checks', BZC[-1])
BZI = pd.concat(BZI, ignore_index=True); BZB = pd.concat(BZB, ignore_index=True)
if min(min(c_['fingerjetfx_identical_to_FPP02'], c_['bozorth3_scores_identical_to_FPP03']) for c_ in BZC) < 0.999: raise RuntimeError('FingerJetFX / BOZORTH3 outputs differ from FPP02 / FPP03')

## 4. Tables: per component, the deployment scenarios, complexity

In [ ]:
def lab(cfg): return f"T4 GPU {cfg['precision']}" if cfg['device'] == 'gpu' else f"CPU {cfg['threads']} thread{'s' if cfg['threads'] > 1 else ''}"
RAW['hardware'] = [lab(r) for r in RAW[['device', 'precision', 'threads']].to_dict('records')]
per_measure = RAW.groupby(['hardware', 'component', 'measure'], sort=False)[['ms_mean', 'ms_median', 'ms_p95', 'batch_ms_per_item']].mean().reset_index()
comp = per_measure.groupby(['hardware', 'component'], sort=False).agg(['mean', 'std']); comp.columns = [f'{a}_{b}' for a, b in comp.columns]; comp = comp.reset_index()
print('Per component, one item per call (mean ± std over the 3 measures of the per-call mean; warm-up excluded):'); display(comp.round(4))
def ms(hw, cname, col='ms_mean'):
    r = comp[(comp.hardware == hw) & (comp.component == cname)]; return (float(r[f'{col}_mean'].iloc[0]), float(r[f'{col}_std'].iloc[0])) if len(r) else (np.nan, np.nan)
cos_ms = float(RAW[RAW.component.str.startswith('cosine')].ms_mean.mean())
lb_ms = float(BZI.letterbox_224_ms.mean()); fx = BZI.groupby('measure').apply(lambda s: s.fingerjetfx_ms.mean() + s.iso_to_xyt_ms.mean()); bz = BZB.groupby('measure').bozorth3_ms_per_pair_amortised.mean()
old_total = fx + bz; OLD = (float(old_total.mean()), float(old_total.std()))
rows = [{'scenario': 'Daily login, old system: minutiae vs minutiae (1:1)', 'method': 'FingerJetFX + BOZORTH3', 'hardware': 'CPU 1 thread', 'template_creation_ms': f'{fx.mean():.2f} ± {fx.std():.2f}',
         'comparison_ms': f'{bz.mean():.3f} ± {bz.std():.3f}', 'probe_to_decision_ms': f'{OLD[0]:.2f} ± {OLD[1]:.2f}', 'speed-up vs old': '1.0×', '_ms': OLD[0]}]
for hw in RAW.hardware.unique():
    a, sa = ms(hw, 'image model (probe image -> embedding)'); b, sb = ms(hw, 'image model + Siamese image tower (first login)')
    rows.append({'scenario': 'Daily login after migration: image vs image (1:1)', 'method': 'letterbox + image model + cosine', 'hardware': hw, 'template_creation_ms': f'{lb_ms + a:.2f} ± {sa:.2f}', 'comparison_ms': f'{cos_ms:.4f}',
                 'probe_to_decision_ms': f'{lb_ms + a + cos_ms:.2f} ± {sa:.2f}', 'speed-up vs old': f'{OLD[0] / (lb_ms + a + cos_ms):.1f}×', '_ms': lb_ms + a + cos_ms})
    rows.append({'scenario': 'First login after migration (once per user)', 'method': 'letterbox + image model + Siamese tower + cosine', 'hardware': hw, 'template_creation_ms': f'{lb_ms + b:.2f} ± {sb:.2f}',
                 'comparison_ms': f'{cos_ms:.4f}', 'probe_to_decision_ms': f'{lb_ms + b + cos_ms:.2f} ± {sb:.2f}', 'speed-up vs old': f'{OLD[0] / (lb_ms + b + cos_ms):.1f}×', '_ms': lb_ms + b + cos_ms})
SCEN = pd.DataFrame(rows)
print('Deployment scenarios (letterbox and comparisons on the CPU; models on the hardware named; warm-up excluded):'); display(SCEN.drop(columns='_ms'))
mig = [{'hardware': hw, 'per_template_single_ms': '{:.2f} ± {:.2f}'.format(*ms(hw, 'migration: minutiae encoder + Siamese tower (per template)')),
        'per_template_batch64_ms': '{:.3f} ± {:.3f}'.format(*ms(hw, 'migration: minutiae encoder + Siamese tower (per template)', 'batch_ms_per_item')),
        'one_million_templates_batch64_minutes': round(ms(hw, 'migration: minutiae encoder + Siamese tower (per template)', 'batch_ms_per_item')[0] * 1e6 / 60000, 1)} for hw in RAW.hardware.unique()]
MIG = pd.DataFrame(mig); print('Migration (offline, once per stored template):'); display(MIG)
CXT = pd.DataFrame(CX) if CX else pd.DataFrame()
if len(CXT): CXT['GFLOPs_per_item'] = CXT.flops_per_item / 1e9; print('Complexity (fp32 graph, batch 1):'); display(CXT.round(3))
FJ = BZI.groupby('dataset')[['fingerjetfx_ms', 'iso_to_xyt_ms', 'letterbox_224_ms', 'minutiae']].mean().reset_index(); print('FingerJetFX per dataset (1 thread):'); display(FJ.round(3))
print('Hardware:', META.cpu_model.iloc[0], '|', os.cpu_count(), 'cores | GPU:', GPU_NAME, '| TensorFlow', META.tensorflow.iloc[0])

## 5. Figure, workbook

In [ ]:
S_ = SCEN[SCEN.scenario.str.startswith('Daily login')]; fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh([f"{m}\n[{h}]" for m, h in zip(S_.method, S_.hardware)], S_._ms, color=['0.55'] + [P.COLORS[0]] * (len(S_) - 1)); ax.set_xscale('log')
for y_, v_ in enumerate(S_._ms): ax.text(v_, y_, f' {v_:.1f} ms', va='center', fontsize=8)
ax.set(xlabel='probe to decision, 1:1 (ms, log scale; warm-up excluded)', title='Daily login after migration: image vs image (ours) vs FingerJetFX + BOZORTH3'); ax.grid(alpha=.3, axis='x')
P._save(fig, FIG / 'daily_login_timing.png')
SHEETS = {'Scenarios': SCEN.drop(columns='_ms'), 'Migration': MIG, 'Per component': comp, 'Complexity': CXT, 'FingerJetFX per dataset': FJ, 'BOZORTH3 per set': BZB,
          'Raw (per measure)': RAW, 'FingerJetFX per image': BZI, 'Checks': META, 'Config': pd.DataFrame([{k: json.dumps(v_, default=str) for k, v_ in CONFIG.items()}]).T.reset_index()}
L.write_workbook(KDIR / 'FPP14_Timing_Results.xlsx', SHEETS)
for n_, df in (('scenarios', SCEN.drop(columns='_ms')), ('migration', MIG), ('per_component', comp), ('complexity', CXT)): df.to_csv(KDIR / f'FPP14_{n_}.csv', index=False)
print('FPP14 complete ->', KDIR)